# Reverse Inference — the REAL test

Does the discovery engine recover the **cause** from a disease **phenotype**, on real data?

It recovers a LINCS-knocked-down gene from its **real experimental signature** — fully non-circular (the signature is real data; the inference uses only the model's network). This is the number that says whether 'give me the phenotype, I'll find the target' actually works.

**No GPU needed** (sparse linear algebra). A standard/high-RAM CPU runtime is fine and starts faster.

Needs two files from earlier work (auto-restored from Drive): `cell_complete.json` (the model) and `lincs_train.npz` (the LINCS table from the transformer notebook).

In [ ]:
# 1) clone + deps + mount Drive
import subprocess, os, sys
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull'])
subprocess.run('pip install -q scipy scikit-learn pandas', shell=True)
from google.colab import drive; drive.mount('/content/drive')
os.makedirs('outputs/orphan', exist_ok=True); print('cwd', os.getcwd())

In [ ]:
# 2) restore the two inputs from Drive (searches known locations, then anywhere under MyDrive)
import shutil, glob, os
OUT='outputs/orphan'
def restore(fname, hints):
    dst=f'{OUT}/{fname}'
    if os.path.exists(dst) and os.path.getsize(dst)>10000: print(fname,'already local'); return True
    for c in hints:
        if os.path.exists(c): shutil.copy(c,dst); print('restored',fname,'from',c); return True
    hits=[h for h in glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True) if os.path.getsize(h)>10000]
    if hits: shutil.copy(hits[0],dst); print('restored',fname,'from',hits[0]); return True
    print('MISSING',fname,'-> run the main build (cell_complete) / transformer notebook (lincs_train) first'); return False
ok1=restore('cell_complete.json', ['/content/drive/MyDrive/cell_model/cell_complete.json','/content/drive/MyDrive/virtual_cell_data/cell_build/cell_complete.json'])
ok2=restore('lincs_train.npz', ['/content/drive/MyDrive/virtual_cell_data/dynamics_transformer/lincs_train.npz'])
print('ready:', ok1 and ok2)

In [ ]:
# 3) THE REAL TEST: recover LINCS-knocked-down genes from their real signatures
import subprocess, sys, json, os
r=subprocess.run([sys.executable,'colab/lincs_reverse_test.py'], capture_output=True, text=True)
print(r.stdout)
if r.stderr.strip(): print('ERR:', r.stderr[-1200:])
p='outputs/orphan/lincs_reverse_test.json'
print('\nRESULT:', json.load(open(p)) if os.path.exists(p) else '(not written)')

In [ ]:
# 4) (reference) sandbox self-consistency + a live example of reverse_infer
import subprocess, sys
print('=== self-consistency check (circular, for reference) ==='); subprocess.run([sys.executable,'colab/compute_reverse_inference.py'])

## Reading the result

- **`recall@10`** = fraction of knocked-down genes ranked in the model's top-10 causal candidates from the signature alone. **`random recall@10`** is the chance rate (~0.0006).
- **If recall@10 is many× the random rate** (e.g. >0.1, i.e. >100× chance), the engine genuinely recovers the cause from real phenotypes → *discovery works*.
- **If it's near chance**, the sandbox's circular success didn't transfer → the inference strategy needs a redesign (that's the honest, data-driven trigger to iterate).
- Watch the **TF vs non-TF** split: recovery is expected to be stronger for genes with an annotated regulon.